# 项目引用与编译性能

学习目标：按依赖图构建多个 TypeScript 项目，观察缓存和声明边界，并用诊断定位编译成本。

前置知识：模块依赖、声明输出、tsconfig、公共接口和命令行构建。

适用版本与条件：TypeScript 7.0.2、Node.js 24.11.0；strict，两个 composite 项目，开启 isolatedDeclarations。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/29-project-references/。

1. [tsconfig.json](scripts/29-project-references/tsconfig.json)：只组织引用关系的解决方案配置。
2. [core/tsconfig.json](scripts/29-project-references/core/tsconfig.json)：基础模块构建及缓存配置。
3. [core/src/index.ts](scripts/29-project-references/core/src/index.ts)：基础模块的公共类型和实现。
4. [app/tsconfig.json](scripts/29-project-references/app/tsconfig.json)：依赖 core 的应用项目。
5. [app/src/main.ts](scripts/29-project-references/app/src/main.ts)：消费 core 构建入口的应用。
6. [isolated-error.ts](scripts/29-project-references/isolated-error.ts)：缺少可独立生成声明条件的反例。

## 1 依赖图决定构建先后

依赖箭头与构建顺序应分别读：app 需要 core，而 core 的契约要先可用。

项目引用（project references）把大项目拆成具有独立配置的部分。本例 app 依赖 core，根配置只列 references 且 files 为空，避免把所有源码再次作为一个项目检查。

普通 tsc -p 只处理指定项目，不负责自动构建其依赖。tsc --build 会遍历引用、判定哪些项目过期，再按依赖顺序构建。引用关系不改写 JavaScript 导入，不会自动安装包。

图的下排反过来按产物供给排列：core 先生成公共声明，app 才能读取这个契约。因此首次操作先 build，再分别 noEmit 检查，最后运行；在依赖输出为空时直接对 app 使用普通 -p，不能检验完整构建链。

![先构建依赖，再构建使用它的项目。上排画依赖方向，下排画首次构建时的产物供给顺序。](image/illustration/29-01-project-reference-build.svg)

图示说明（依据篇末官方文档自绘）：图描述本例首次构建；后续哪些项目可以跳过，要看 --build 的过期判断。

对照下面 references，再用 build:29 观察先 core 后 app 的产物；运行时仍需核对 core 的 JavaScript 导入路径。

以下片段来自 tsconfig.json。

```json
{
  "compilerOptions": {
    "strict": true
  },
  "files": [],
  "references": [
    {
      "path": "./core"
    },
    {
      "path": "./app"
    }
  ]
}
```

Step 1：按依赖图构建。

```bash
npm run build:29
# 先生成 core/dist，再生成 app/dist。
```

Step 2：检查已具备依赖声明的两个项目。

```bash
npm run check:29
# 各项目 noEmit 检查，无意外诊断。
```

Step 3：运行应用构建物。

```bash
npm run run:29
# total 10。
```

## 2 composite 与声明边界

被引用项目开启 composite，要求实现文件全部由 files 或 include 纳入，并提供声明输出。本例明确 rootDir、outDir 和 include，避免意外把同级测试或临时文件合入工程。

declarationMap 为跨项目定义跳转提供映射。它不影响 JavaScript 的运行导入；编辑器能否跳转还取决于对应源码与版本支持。app 显式引用 core 项目，并从 core/dist/index.js 导入，检查时解析相邻声明，运行时加载真正的 JavaScript。

以下片段来自 core/tsconfig.json。

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "NodeNext",
    "moduleResolution": "NodeNext",
    "strict": true,
    "lib": [
      "ES2025"
    ],
    "types": [
      "node"
    ],
    "composite": true,
    "declaration": true,
    "declarationMap": true,
    "incremental": true,
    "isolatedDeclarations": true,
    "rootDir": "src",
    "outDir": "dist",
    "tsBuildInfoFile": "./dist/build.tsbuildinfo",
    "noEmitOnError": true
  },
  "include": [
    "src/**/*.ts"
  ]
}
```

以下片段来自 app/tsconfig.json。

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "NodeNext",
    "moduleResolution": "NodeNext",
    "strict": true,
    "lib": [
      "ES2025"
    ],
    "types": [
      "node"
    ],
    "composite": true,
    "declaration": true,
    "declarationMap": true,
    "incremental": true,
    "isolatedDeclarations": true,
    "rootDir": "src",
    "outDir": "dist",
    "tsBuildInfoFile": "./dist/build.tsbuildinfo",
    "noEmitOnError": true
  },
  "include": [
    "src/**/*.ts"
  ],
  "references": [
    {
      "path": "../core"
    }
  ]
}
```

## 3 让输出路径与模块路径对应

core 的接口和 total 是公共契约，total 明确返回 number。app/src 与 app/dist 深度相同，因此 ../../core/dist/index.js 在源码和输出中都指向预期目录；references 不负责修正错误的相对路径。

跨包发布时还要设计 package.json 入口；本章只观察同工作目录内两个项目的构建关系。

以下片段来自 core/src/index.ts。

```typescript
export interface Item { price: number }
export function total(items: readonly Item[]): number {
  return items.reduce((sum, item) => sum + item.price, 0);
}
```

以下片段来自 app/src/main.ts。

```typescript
import { total } from "../../core/dist/index.js";
const amount = total([{ price: 4 }, { price: 6 }]);
console.log("total", amount); // total 10。
```

## 4 增量缓存、重新构建与清理

incremental 把后续构建可复用的信息保存在 .tsbuildinfo；本例用 tsBuildInfoFile 显式放到各项目 dist 中，缓存属于工具输出，不是运行时代码。不要编辑内部内容，也不要把旧编译器缓存的存在当作当前检查通过。

再次 --build 时，已是最新的项目可以跳过；--force 要求重建，--verbose 说明判定原因。修改公共类型可能使下游需要重查，纯内部变化也不能简单按“文件变了”推断所有项目一定如何处理，应观察构建日志。

较新的 --build 可在上游错误时继续处理其他项目，不能沿用早期手册“总是立刻停止”的描述。本例各项目显式 noEmitOnError，构建命令再使用 stopBuildOnErrors 跳过错误项目的下游。

Step 1：观察无源码变化时的构建判定。

```bash
npm run build:29
# --verbose 可显示项目已是最新；具体时间戳不固定。
```

Step 2：预览将被清理的构建输出。

```bash
npm run clean:29:dry
# --clean --dry 只列待删除输出。
```

Step 3：清理本章构建物和缓存。

```bash
npm run clean:29
# 下一次运行前需重新 build:29。
```

## 5 isolatedDeclarations 的用途和代价

isolatedDeclarations 要求导出声明具有足够的局部类型信息，以便工具不依赖跨文件类型推断也能生成声明。它需要 declaration 或 composite，但不是 isolatedModules，也不是关闭检查或立即提高性能的开关。

本例 total 标明参数和返回类型；普通局部变量仍可推断。并非所有导出都必须机械加标注，简单字面量有可直接推导的情况；下面调用 join 的表达式在本地 TS 7.0.2 下报告 TS9013，修复是明确 makeLabel 的返回类型为 string。

以下片段来自 isolated-error.ts。

```typescript
export function makeLabel() { return ["Ada"].join(""); } // TS9013：此返回表达式不能仅凭局部语法推导声明。
```

以下片段来自 tsconfig.errors.json。

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "NodeNext",
    "moduleResolution": "NodeNext",
    "strict": true,
    "lib": [
      "ES2025"
    ],
    "types": [],
    "declaration": true,
    "isolatedDeclarations": true,
    "noEmit": true
  },
  "files": [
    "isolated-error.ts"
  ]
}
```

Step 1：核对独立声明条件的反例。

```bash
npm run errors:29
# 退出 1，包含 TS9013。
```

## 6 诊断解析成本与检查成本

extendedDiagnostics 显示文件、类型、实例化、内存及阶段耗时，适合先区分解析、检查与生成成本。traceResolution 记录模块搜索和最终文件，用于解释为何读到了错误声明或过多依赖；它不是类型执行跟踪，也不是运行时性能分析。

同样源码的冷构建、缓存构建和 noEmit 检查不能直接拿毫秒数互比。要比较编译成本，应固定机器、工具版本、输入范围与缓存条件；下面强制构建仅用于获得本次诊断，不给出固定耗时或提升倍数。

Step 1：强制构建并输出阶段诊断。

```bash
npm run diagnose:29
# 查看 Check time、Parse time、Memory used 等项目；实际数值由机器决定。
```

Step 2：查看 app 的模块解析过程。

```bash
npm run trace:29
# 可找到 ../../core/dist/index.js 解析到 core/dist/index.d.ts 的记录。
```

## 7 控制类型复杂度与编辑器负担

性能处理先检查 include 范围、重复项目和无关类型包，再调查大的联合、交叉和递归条件类型。稳定的公共接口和可命名复用的复杂类型有助于控制重复计算；不用更深的类型技巧来证明“高级”。

编辑器慢还可能来自加载范围、插件或语言服务版本，与命令行 tsc 的瓶颈不完全一致。TypeScript 7 可并行处理检查和项目引用，checkers 与 builders 是调节选项，不保证线程越多越快，还会增加内存压力；本小例保持默认，不据此推断大型工程收益。

JavaScript 的运行耗时属于生成程序的工作；即使 tsc 检查更快，total 的运行算法并没有自动变快。

## 本章小结

- references 描述依赖图，--build 负责排序与过期判断，普通 -p 不会构建依赖。
- composite、声明映射和独立声明条件帮助表达可构建的边界。
- 缓存与诊断必须在明确条件下解读，编译耗时和程序运行耗时分开。

## 练习

1. 给 Item 增加必需数量字段 quantity，并让 total 计算价格乘数量；更新 app 后构建运行，输入 4×2 与 6×1 应为 14。
2. 连续运行两次 build:29，记录第二次哪些项目被跳过；清理后再构建，确认两个项目重新产生声明和 JavaScript。
3. 给 makeLabel 添加返回类型 string，确认 TS9013 消失；说明为什么这不是关闭类型检查。
4. 在 trace:29 输出中定位 core 的声明路径，核对它与运行导入路径的扩展名差别。

### 提示

1. 同步修改 Item、total 和 app 的两项输入。
2. 第二次构建前不改源码或配置；清理后再次构建才比较完整产物。
3. 给函数写返回标注，函数体仍接受检查。
4. 对照 .d.ts 与 .js 各自面向的工具。

### 参考解析

1. Item 增加 `quantity: number`，累加项改为 `item.price * item.quantity`，输入分别提供 quantity: 2 和 quantity: 1；输出 total 14。
2. 无变化时日志应说明项目已是最新；清理后两个项目重新生成声明与 JavaScript。缓存判定依赖实际文件状态，不预设固定耗时。
3. `makeLabel(): string` 为独立声明生成提供局部返回信息；函数实现仍需满足 string，因此并未关闭类型检查。
4. 检查器解析 core/dist/index.d.ts，Node 运行导入仍指向 core/dist/index.js；references 不改写这个说明符。

## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [Project References](https://www.typescriptlang.org/docs/handbook/project-references.html) 的 references、composite、declarationMap、构建和清理；[incremental](https://www.typescriptlang.org/tsconfig/incremental.html)、[isolatedDeclarations](https://www.typescriptlang.org/tsconfig/isolatedDeclarations.html)、[5.5 / Isolated Declarations](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-5-5.html#isolated-declarations)、[5.6 / Intermediate Errors](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-5-6.html#allow---build-with-intermediate-errors)；[extendedDiagnostics](https://www.typescriptlang.org/tsconfig/extendedDiagnostics.html)、[traceResolution](https://www.typescriptlang.org/tsconfig/traceResolution.html)：构建条件和诊断用途。 |
| GitHub / Microsoft | [Performance](https://github.com/microsoft/TypeScript/wiki/Performance) 的命名复杂类型、项目范围、项目引用和诊断方法；[TypeScript-Website / Compiler Options](https://github.com/microsoft/TypeScript-Website/blob/v2/packages/documentation/copy/en/project-config/Compiler%20Options.md) 的 --stopBuildOnErrors 条目：编译负担及上游失败时跳过下游。 |
| Microsoft Developer Blogs | [TypeScript 7.0 / Custom Scaling](https://devblogs.microsoft.com/typescript/announcing-typescript-7-0/#custom-scaling-parallelization-and-controls)：checkers、builders 和并行成本。 |
